# **Speed calculation (Romain)**

Import the library used in this notebook + set the path

In [34]:
import cv2
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "ua-detrac" / "DETRAC-Images"  / "DETRAC-Images"  
DATA_DIR = DATA_DIR.resolve()
VIDEO_OUTPUT_DIR = PROJECT_ROOT / "data_processed" / "videos"
VIDEO_OUTPUT_DIR=VIDEO_OUTPUT_DIR.resolve()
TRAIN_ANNOTATIONS_DIR = PROJECT_ROOT / "data" / "ua-detrac" / "DETRAC-Train-Annotations-XML" / "DETRAC-Train-Annotations-XML"
TRAIN_ANNOTATIONS_DIR = TRAIN_ANNOTATIONS_DIR.resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("Exists:", DATA_DIR.exists())
print("VIDEO_OUTPUT_DIR:", VIDEO_OUTPUT_DIR)

FPS = 25 # Frames per second for video playback
MAX_FRAMES = 2000  # Maximum number of frames to read/play just in case

PROJECT_ROOT: e:\ICV-Project
DATA_DIR: E:\ICV-Project\data\ua-detrac\DETRAC-Images\DETRAC-Images
Exists: True
VIDEO_OUTPUT_DIR: E:\ICV-Project\data_processed\videos


Reading of the frames of all the videos

In [35]:
sequences = sorted([p for p in DATA_DIR.iterdir() if p.is_dir() and p.name.startswith("MVI_")])
print("Nb sequences:", len(sequences))
print("Example:", sequences[0])

imgs = sorted(Path(sequences[0]).glob("img*.jpg"))[:500]
frames = [cv2.imread(str(p)) for p in imgs]  # préchargement

Nb sequences: 100
Example: E:\ICV-Project\data\ua-detrac\DETRAC-Images\DETRAC-Images\MVI_20011


Display the video with bonding boxes (*Deprecated - do not use*)

In [36]:
# # Read video
# video_name = 'cctv052x2004080516x01638.avi'  # video file name
# video_path = os.path.join(VIDEO_DIR, video_name)  # build full path

# video = cv2.VideoCapture(video_path)  # open video file

# # Visualize video
# while True:
#     ret, frame = video.read()  # read one frame
#     if not ret:                # stop if no frame is returned (end of video)
#         break
    
#     cv2.imshow('frame', frame)  # display current frame

#     for (x, y, w, h) in face_rects:                         # Loop over detections (for each face)
#     cv2.rectangle(face_img, (x, y), (x+w, y+h),
#                     (255, 29, 0), 5)                     # Draw a rectangle around each vehicles



#     key = cv2.waitKey(5)              # wait ~5 ms
#     if key == ord('q'):                     # stop if 'q' key is pressed
#         break
        

# video.release()                 # release video object
# cv2.destroyAllWindows()         # close all OpenCV windows

Display the images with bonding boxes (*Deprecated - do not use*)

In [37]:
# import os
# from PIL import Image, ImageDraw, ImageFont

# IMG_DIR = "data/vd8c/images"
# LBL_DIR = "data/vd8c/labels"

# img_path = os.path.join(IMG_DIR, fname)
# label_name = os.path.splitext(fname)[0] + ".txt"
# txt_path = os.path.join(LBL_DIR, label_name)

# img = Image.open(img_path).convert("RGB")
# W, H = img.size
# draw = ImageDraw.Draw(img)

# with open(txt_path, "r", encoding="utf-8") as f:
#     for line in f:
#         parts = line.strip().split()
#         if len(parts) != 5:
#             continue

#         class_id_str, x, y, w, h = parts
#         x, y, w, h = map(float, (x, y, w, h))

#         # YOLO -> coins en pixels
#         x_min = (x - w / 2) * W
#         y_min = (y - h / 2) * H
#         x_max = (x + w / 2) * W
#         y_max = (y + h / 2) * H

#         # clamp dans l'image
#         x_min = max(0, min(W - 1, x_min))
#         y_min = max(0, min(H - 1, y_min))
#         x_max = max(0, min(W - 1, x_max))
#         y_max = max(0, min(H - 1, y_max))

#         # rectangle
#         draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=2)

#         # texte = class_id (sans dictionnaire)
#         label = class_id_str

#         # position du texte (au-dessus si possible)
#         tx, ty = x_min, max(0, y_min - 18)

#         # fond du texte pour lisibilité
#         text_bbox = draw.textbbox((tx, ty), label, font=font)
#         draw.rectangle(text_bbox, fill="red")
#         draw.text((tx, ty), label, fill="white", font=font)

# img.show()


# New version of Romain's notebook (17/01/2026)

### Rebuild the video (CHANGE HERE TO SELECT THE VIDEO SEQUENCE TO REBUILD)

Functions for writing and play the video

In [38]:
def write_avi(
    frames,
    output_path,
    fps=25,
    codec="XVID"
):
    """
    frames : list[np.ndarray] (BGR OpenCV)
    output_path : str ou Path, ex: "output.avi"
    fps : int
    codec : FOURCC, ex: XVID, MJPG
    """

    assert len(frames) > 0, "Liste de frames vide"

    h, w = frames[0].shape[:2]

    fourcc = cv2.VideoWriter_fourcc(*codec)
    writer = cv2.VideoWriter(
        str(output_path),
        fourcc,
        fps,
        (w, h)
    )

    if not writer.isOpened():
        raise RuntimeError("Impossible d'ouvrir le VideoWriter")

    for i, frame in enumerate(frames):
        if frame is None:
            raise RuntimeError(f"Frame {i} invalide")
        writer.write(frame)

    writer.release()
    print(f"Vidéo écrite : {output_path}")


def play_video(video_path, window_name="Video", fps=None):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Impossible d’ouvrir la vidéo : {video_path}")

    # FPS depuis le fichier si non imposé
    if fps is None:
        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps <= 0:
            fps = 25  # fallback

    delay_ms = int(1000 / fps)

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            cv2.imshow(window_name, frame)

            key = cv2.waitKey(delay_ms) & 0xFF
            if key == ord('q') or key == 27:  # q ou ESC
                break

            # fermeture via la croix
            if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
                break
    finally:
        cap.release()
        cv2.destroyWindow(window_name)
        cv2.waitKey(1)

In [39]:
# Please ignore the commented code below if you don't need it (video rebuilt) - Ctrl+/ to uncomment

i=0
n=len(sequences)

for seq in sequences:
    video_name = seq.name
    out = Path(VIDEO_OUTPUT_DIR, f"{video_name}.avi")

    # --- création vidéo si absente ---
    if not out.exists():
        print(f"[WRITE] {video_name}")

        imgs = sorted(seq.glob("img*.jpg"))
        if MAX_FRAMES is not None:
            imgs = imgs[:MAX_FRAMES]

        if len(imgs) == 0:
            print(f"[SKIP] aucune image pour {video_name}")
            continue

        frames = []
        for p in imgs:
            img = cv2.imread(str(p))
            if img is None:
                raise RuntimeError(f"Image illisible : {p}")
            frames.append(img)

        write_avi(
            frames=frames,
            output_path=out,
            fps=FPS,
            codec="XVID"
        )

    #else:
        #print(f"[EXISTS] {video_name}")

    print(video_name,'rebuilt.',"Progress :", i+1, "/",n)
    i+=1


MVI_20011 rebuilt. Progress : 1 / 100
MVI_20012 rebuilt. Progress : 2 / 100
MVI_20032 rebuilt. Progress : 3 / 100
MVI_20033 rebuilt. Progress : 4 / 100
MVI_20034 rebuilt. Progress : 5 / 100
MVI_20035 rebuilt. Progress : 6 / 100
MVI_20051 rebuilt. Progress : 7 / 100
MVI_20052 rebuilt. Progress : 8 / 100
MVI_20061 rebuilt. Progress : 9 / 100
MVI_20062 rebuilt. Progress : 10 / 100
MVI_20063 rebuilt. Progress : 11 / 100
MVI_20064 rebuilt. Progress : 12 / 100
MVI_20065 rebuilt. Progress : 13 / 100
MVI_39031 rebuilt. Progress : 14 / 100
MVI_39051 rebuilt. Progress : 15 / 100
MVI_39211 rebuilt. Progress : 16 / 100
MVI_39271 rebuilt. Progress : 17 / 100
MVI_39311 rebuilt. Progress : 18 / 100
MVI_39361 rebuilt. Progress : 19 / 100
MVI_39371 rebuilt. Progress : 20 / 100
MVI_39401 rebuilt. Progress : 21 / 100
MVI_39501 rebuilt. Progress : 22 / 100
MVI_39511 rebuilt. Progress : 23 / 100
MVI_39761 rebuilt. Progress : 24 / 100
MVI_39771 rebuilt. Progress : 25 / 100
MVI_39781 rebuilt. Progress : 26 /

To play a video

In [50]:
video_name = "MVI_20011.avi"

video_path=VIDEO_OUTPUT_DIR / video_name
play_video(video_path, window_name=video_name)

## Calculation of vehicles speed

Import des libraries

In [43]:
import xml.etree.ElementTree as ET
from collections import defaultdict
from pathlib import Path
import numpy as np

Chargement des trajectoires depuis le XML DETRAC

In [44]:
def load_vehicle_tracks_from_detrac_xml(xml_file_path):
    """
    Lit un fichier XML UA-DETRAC et reconstruit les trajectoires véhicule.

    Paramètre
    ---------
    xml_file_path : Path ou str
        Chemin vers le fichier XML de la séquence (ex: MVI_20011.xml)

    Retour
    ------
    vehicle_tracks : dict
        vehicle_tracks[vehicle_id] = list of (frame_number, center_x, center_y)
    """

    # Chargement du XML
    xml_root = ET.parse(xml_file_path).getroot()

    # Dictionnaire des trajectoires par véhicule
    vehicle_tracks = defaultdict(list)

    # Parcours de toutes les frames de la séquence
    for frame_node in xml_root.findall(".//frame"):
        frame_number = int(frame_node.attrib["num"])

        # Parcours de tous les véhicules visibles dans la frame
        for target_node in frame_node.findall(".//target_list/target"):
            vehicle_id = int(target_node.attrib["id"])

            # Lecture de la bounding box
            box_node = target_node.find("box")
            left   = float(box_node.attrib["left"])
            top    = float(box_node.attrib["top"])
            width  = float(box_node.attrib["width"])
            height = float(box_node.attrib["height"])

            # Centre géométrique de la bounding box
            center_x = left + width  / 2.0
            center_y = top  + height / 2.0

            # Stockage de la position pour ce véhicule
            vehicle_tracks[vehicle_id].append(
                (frame_number, center_x, center_y)
            )

    # Sécurité : trier les points par frame croissante
    for vehicle_id in vehicle_tracks:
        vehicle_tracks[vehicle_id].sort(key=lambda p: p[0])

    return vehicle_tracks


Calcul de la vitesse image-plane (px/s)

In [45]:
def compute_vehicle_speed_px_per_second(
    vehicle_track,
    frames_per_second=25
):
    """
    Calcule la vitesse instantanée d'un véhicule à partir de sa trajectoire.

    Paramètres
    ----------
    vehicle_track : list of (frame_number, x, y)
        Trajectoire triée par frame
    frames_per_second : int
        FPS de la vidéo (DETRAC = 25)

    Retour
    ------
    speed_frames : list[int]
        Numéros de frame associés aux vitesses calculées
    speeds_px_per_s : np.ndarray
        Vitesse en pixels par seconde
    """

    # Une vitesse nécessite au moins deux positions
    if len(vehicle_track) < 2:
        return [], np.array([])

    # Extraction des données
    frame_numbers = np.array([p[0] for p in vehicle_track], dtype=float)
    x_positions   = np.array([p[1] for p in vehicle_track], dtype=float)
    y_positions   = np.array([p[2] for p in vehicle_track], dtype=float)

    # Différences temporelles (gère frames manquantes)
    delta_time = np.diff(frame_numbers) / frames_per_second

    # Déplacements spatiaux
    delta_x = np.diff(x_positions)
    delta_y = np.diff(y_positions)

    # Distance parcourue entre frames (pixels)
    distance_pixels = np.hypot(delta_x, delta_y)

    # Vitesse instantanée (px/s)
    speeds_px_per_s = distance_pixels / delta_time

    # Les vitesses sont associées à la frame courante
    speed_frames = frame_numbers[1:].astype(int).tolist()

    return speed_frames, speeds_px_per_s


Lissage (obligatoire en pratique)

In [46]:
def smooth_speed_with_moving_average(
    speed_array,
    window_size=5
):
    """
    Applique une moyenne glissante pour réduire le bruit.

    Paramètres
    ----------
    speed_array : np.ndarray
        Vitesses brutes
    window_size : int
        Taille de la fenêtre de lissage

    Retour
    ------
    np.ndarray
        Vitesses lissées
    """

    if len(speed_array) < window_size:
        return speed_array

    kernel = np.ones(window_size) / window_size
    return np.convolve(speed_array, kernel, mode="same")


Conversion optionnelle vers km/h (si calibration connue)

In [47]:
def convert_speed_px_s_to_kmh(
    speed_px_per_s,
    meters_per_pixel
):
    """
    Convertit une vitesse px/s en km/h.

    Paramètres
    ----------
    speed_px_per_s : np.ndarray
        Vitesse en pixels par seconde
    meters_per_pixel : float
        Échelle métrique caméra

    Retour
    ------
    np.ndarray
        Vitesse en km/h
    """
    speed_m_per_s = speed_px_per_s * meters_per_pixel
    return speed_m_per_s * 3.6


Exemple complet sur MVI_20011

In [48]:
xml_path = Path(TRAIN_ANNOTATIONS_DIR) / "MVI_20011.xml"

# Chargement des trajectoires
vehicle_tracks = load_vehicle_tracks_from_detrac_xml(xml_path)

# Sélection d'un véhicule
vehicle_id = 1
fps = 25

# Calcul vitesse brute
speed_frames, speed_px_s = compute_vehicle_speed_px_per_second(
    vehicle_tracks[vehicle_id],
    frames_per_second=fps
)

# Lissage
speed_px_s_smooth = smooth_speed_with_moving_average(
    speed_px_s,
    window_size=5
)

print("Vitesse px/s (10 premières) :", speed_px_s[:10])
print("Vitesse px/s lissée :", speed_px_s_smooth[:10])


Vitesse px/s (10 premières) : [171.47457902 220.73209157 202.04060545 237.75203732]
Vitesse px/s lissée : [171.47457902 220.73209157 202.04060545 237.75203732]
